# Deep CFR GPU Fitting Benchmark

This notebook isolates neural fitting from traversal and exact evaluation. It measures:

1. The current `DeepCFRTrainer._train_model` path, including replay sampling and CPU-to-device copies.
2. A device-resident upper bound where one batch is already on the target device.

The gap between those measurements tells us whether GPU fitting is limited by matrix compute or by the current replay/data pipeline.

In [ ]:
from pathlib import Path
import random
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'liars_poker').exists() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from liars_poker.algo.deep_cfr import DeepCFRTrainer
from liars_poker.core import GameSpec
from liars_poker.infoset import CALL
from liars_poker.policies.neural import NeuralMLP

torch.set_float32_matmul_precision('high')

print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('CPU threads:', torch.get_num_threads())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('CUDA runtime:', torch.version.cuda)
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    print(f'GPU memory: {free_bytes / 2**30:.1f} / {total_bytes / 2**30:.1f} GiB free')

## Representative Replay Data

The records below have the same shapes, masks, weights, and storage format used by Deep CFR. Their numerical values are synthetic because this benchmark is about fitting throughput, not policy quality.

In [ ]:
spec = GameSpec(
    ranks=4,
    suits=4,
    hand_size=3,
    claim_kinds=('RankHigh', 'Pair', 'Trips'),
    suit_symmetry=True,
)

REPLAY_RECORDS = 50_000
FIT_STEPS = 100
WARMUP_STEPS = 10
REPEATS = 3

# This compact screen should finish quickly while covering the current baseline
# and a larger network/batch that should benefit more from a GPU.
QUICK_CONFIGS = [
    ((256, 256), 256),
    ((256, 256), 1024),
    ((256, 256), 4096),
    ((512, 512), 1024),
    ((512, 512), 4096),
]

print(spec.to_short_str())
print('fit steps per timing:', FIT_STEPS)
print('replay records per player/buffer:', REPLAY_RECORDS)

In [ ]:
def synchronize(device):
    if torch.device(device).type == 'cuda':
        torch.cuda.synchronize()


def make_replay_arrays(trainer, records, seed):
    rng = np.random.default_rng(seed)
    input_dim = trainer.encoder.input_dim
    action_dim = trainer.encoder.action_dim

    features = rng.normal(0.0, 1.0, size=(records, input_dim)).astype(np.float32)
    masks = np.zeros((records, action_dim), dtype=bool)

    # Generate masks from real legal-action sets at randomly selected last claims.
    last_claims = rng.integers(-1, trainer.encoder.k, size=records)
    for row, last_claim in enumerate(last_claims):
        legal = trainer.rules.legal_actions_from_last(None if last_claim < 0 else int(last_claim))
        cols = [0 if action == CALL else action + 1 for action in legal]
        masks[row, cols] = True

    advantages = rng.normal(0.0, 0.6, size=(records, action_dim)).astype(np.float32)
    advantages[~masks] = 0.0

    strategy_mass = rng.gamma(1.0, 1.0, size=(records, action_dim)).astype(np.float32)
    strategy_mass[~masks] = 0.0
    strategies = strategy_mass / strategy_mass.sum(axis=1, keepdims=True)

    weights = rng.integers(1, 1001, size=records).astype(np.float32)
    return features, advantages, strategies, masks, weights


def load_buffer(buffer, features, targets, masks, weights):
    n = len(features)
    buffer.features[:n] = features
    buffer.targets[:n] = targets
    buffer.legal_masks[:n] = masks
    buffer.weights[:n] = weights
    buffer.size = n
    buffer.seen = n


def build_fitting_trainer(device, hidden_sizes, batch_size, seed):
    trainer = DeepCFRTrainer(
        spec,
        hidden_sizes=hidden_sizes,
        device=device,
        seed=seed,
        batch_size=batch_size,
        advantage_train_steps=0,
        strategy_train_steps=0,
        advantage_positive_weight=0.5,
        advantage_buffer_capacity=REPLAY_RECORDS,
        strategy_buffer_capacity=REPLAY_RECORDS,
    )
    features, advantages, strategies, masks, weights = make_replay_arrays(
        trainer, REPLAY_RECORDS, seed + 10_000
    )
    for pid in (0, 1):
        load_buffer(trainer.advantage_buffers[pid], features, advantages, masks, weights)
        load_buffer(trainer.strategy_buffers[pid], features, strategies, masks, weights)
    return trainer

## End-to-End Fitting Path

This calls the trainer's real fitting implementation. It therefore includes NumPy replay sampling, tensor construction, host-to-device transfer, forward/backward passes, and Adam updates.

In [ ]:
def benchmark_current_fit(device, hidden_sizes, batch_size, *, repeat, steps=FIT_STEPS):
    trainer = build_fitting_trainer(device, hidden_sizes, batch_size, seed=1000 + repeat)

    # Warm up PyTorch kernels and allocator state before measuring.
    trainer._train_model(
        trainer.advantage_nets[0],
        trainer.advantage_optimizers[0],
        trainer.advantage_buffers[0],
        WARMUP_STEPS,
        strategy_loss=False,
    )
    trainer._train_model(
        trainer.strategy_nets[0],
        trainer.strategy_optimizers[0],
        trainer.strategy_buffers[0],
        WARMUP_STEPS,
        strategy_loss=True,
    )

    synchronize(device)
    start = time.perf_counter()
    advantage_loss = trainer._train_model(
        trainer.advantage_nets[0],
        trainer.advantage_optimizers[0],
        trainer.advantage_buffers[0],
        steps,
        strategy_loss=False,
    )
    synchronize(device)
    advantage_s = time.perf_counter() - start

    synchronize(device)
    start = time.perf_counter()
    strategy_loss = trainer._train_model(
        trainer.strategy_nets[0],
        trainer.strategy_optimizers[0],
        trainer.strategy_buffers[0],
        steps,
        strategy_loss=True,
    )
    synchronize(device)
    strategy_s = time.perf_counter() - start

    return {
        'device': str(device),
        'architecture': 'x'.join(map(str, hidden_sizes)),
        'batch size': batch_size,
        'repeat': repeat,
        'steps': steps,
        'advantage s': advantage_s,
        'strategy s': strategy_s,
        'estimated full iteration fit s': 2 * (advantage_s + strategy_s),
        'advantage updates / s': steps / advantage_s,
        'strategy updates / s': steps / strategy_s,
        'advantage loss': advantage_loss,
        'strategy loss': strategy_loss,
    }


devices = ['cpu'] + (['cuda'] if torch.cuda.is_available() else [])
fit_rows = []
for hidden_sizes, batch_size in QUICK_CONFIGS:
    for target_device in devices:
        for repeat in range(REPEATS):
            print(target_device, hidden_sizes, batch_size, 'repeat', repeat + 1)
            fit_rows.append(
                benchmark_current_fit(
                    target_device,
                    hidden_sizes,
                    batch_size,
                    repeat=repeat,
                )
            )

fit_raw_df = pd.DataFrame(fit_rows)
fit_summary_df = (
    fit_raw_df
    .groupby(['device', 'architecture', 'batch size'], as_index=False)
    .agg(
        repeats=('repeat', 'size'),
        median_advantage_s=('advantage s', 'median'),
        median_strategy_s=('strategy s', 'median'),
        median_full_iteration_fit_s=('estimated full iteration fit s', 'median'),
        median_advantage_updates_per_s=('advantage updates / s', 'median'),
        median_strategy_updates_per_s=('strategy updates / s', 'median'),
    )
    .sort_values(['architecture', 'batch size', 'device'])
)

display(fit_summary_df)
display(fit_raw_df)

In [ ]:
if {'cpu', 'cuda'}.issubset(set(fit_summary_df['device'])):
    speedup_df = (
        fit_summary_df
        .pivot(index=['architecture', 'batch size'], columns='device', values='median_full_iteration_fit_s')
        .reset_index()
    )
    speedup_df['GPU speedup'] = speedup_df['cpu'] / speedup_df['cuda']
    display(speedup_df.sort_values('GPU speedup', ascending=False))

    fig, ax = plt.subplots(figsize=(10, 5))
    labels = speedup_df['architecture'] + ', batch=' + speedup_df['batch size'].astype(str)
    ax.bar(labels, speedup_df['GPU speedup'])
    ax.axhline(1.0, color='black', linewidth=1)
    ax.set_ylabel('CPU time / CUDA time')
    ax.set_title('End-to-end fitting speedup')
    ax.tick_params(axis='x', rotation=35)
    plt.tight_layout()
    plt.show()
else:
    print('CUDA was unavailable, so only the CPU baseline was measured.')

## Device-Resident Compute Ceiling

The next benchmark keeps a fixed batch on the target device. It is not the current trainer behavior; it estimates how fast fitting could become if replay sampling supplied device-resident batches efficiently.

In [ ]:
def benchmark_resident_fit(device, hidden_sizes, batch_size, *, repeat, steps=FIT_STEPS):
    device = torch.device(device)
    trainer = build_fitting_trainer(device, hidden_sizes, batch_size, seed=3000 + repeat)
    features, targets, masks, weights = trainer.advantage_buffers[0].sample(batch_size, trainer.rng)

    x = torch.from_numpy(features).to(device)
    y = torch.from_numpy(targets).to(device)
    mask = torch.from_numpy(masks).to(device)
    weight = torch.from_numpy(weights).to(device)
    weight = weight / weight.mean().clamp_min(1e-8)

    model = trainer.advantage_nets[0]
    optimizer = trainer.advantage_optimizers[0]

    def update():
        pred = model(x)
        mask_float = mask.float()
        entry_weight = mask_float * (1.0 + 0.5 * (y > 1e-6).float())
        squared = (pred - y).square() * entry_weight
        per_sample = squared.sum(dim=1) / entry_weight.sum(dim=1).clamp_min(1.0)
        loss = (per_sample * weight).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        return loss

    for _ in range(WARMUP_STEPS):
        update()
    synchronize(device)

    start = time.perf_counter()
    loss = None
    for _ in range(steps):
        loss = update()
    synchronize(device)
    elapsed = time.perf_counter() - start

    return {
        'device': device.type,
        'architecture': 'x'.join(map(str, hidden_sizes)),
        'batch size': batch_size,
        'repeat': repeat,
        'steps': steps,
        'resident advantage s': elapsed,
        'resident updates / s': steps / elapsed,
        'loss': float(loss.detach().cpu()),
    }


resident_rows = []
for hidden_sizes, batch_size in QUICK_CONFIGS:
    for target_device in devices:
        for repeat in range(REPEATS):
            resident_rows.append(
                benchmark_resident_fit(
                    target_device,
                    hidden_sizes,
                    batch_size,
                    repeat=repeat,
                )
            )

resident_raw_df = pd.DataFrame(resident_rows)
resident_summary_df = (
    resident_raw_df
    .groupby(['device', 'architecture', 'batch size'], as_index=False)
    .agg(
        median_resident_advantage_s=('resident advantage s', 'median'),
        median_resident_updates_per_s=('resident updates / s', 'median'),
    )
    .sort_values(['architecture', 'batch size', 'device'])
)
display(resident_summary_df)

In [ ]:
pipeline_df = fit_summary_df.merge(
    resident_summary_df,
    on=['device', 'architecture', 'batch size'],
    how='left',
)
pipeline_df['sampling + transfer overhead factor'] = (
    pipeline_df['median_advantage_s'] / pipeline_df['median_resident_advantage_s']
)
display(
    pipeline_df[[
        'device',
        'architecture',
        'batch size',
        'median_advantage_s',
        'median_resident_advantage_s',
        'sampling + transfer overhead factor',
    ]].sort_values(['architecture', 'batch size', 'device'])
)

## Reading the Results

- A large **GPU speedup** in the end-to-end table means the existing trainer immediately benefits from CUDA.
- A much faster **device-resident** result than the current CUDA path means replay sampling, tensor creation, and host-to-device copies are hiding much of the GPU's potential.
- Larger networks and batches should generally improve GPU utilization. Tiny MLPs or batches may remain CPU-competitive because kernel-launch and transfer overhead dominate.
- The estimated full-iteration fitting time assumes two player-specific advantage fits and two player-specific strategy fits. Traversal and exact evaluation are intentionally absent.

The next engineering decision should follow the measured bottleneck. If current CUDA fitting is already close to the resident ceiling, focus on batched traversal. If the gap is large, make replay sampling return pinned batches and overlap asynchronous transfer with GPU compute.